### 🔥 Temperature Prediction

This notebook forecasts Australia's national mean temperature using historical climate trends and machine learning. Results will help assess future climate risk and habitability.


In [ ]:
import pandas as pd
import os

FINAL_DATASET_DIR = r"d:\PersonalProjects\WeatherPredictionAustralia\data\final_dataset"

df = pd.read_csv(os.path.join(FINAL_DATASET_DIR, "australia_climate_ml_ready.csv"))
print(df)

### 📈 Predicting Each Climate Trend Individually

To maximize historical coverage and forecast reliability, we will first model each major climate trend—beginning with temperature—using its own full time series, rather than only the years shared across all datasets.


In [ ]:
TEMP_DATASET_DIR = r"d:\PersonalProjects\WeatherPredictionAustralia\data\temp"
temp_df = pd.read_csv(os.path.join(TEMP_DATASET_DIR, "aus_mean_temp.csv"))
print(temp_df)

In [ ]:
# Decade to "Year" conversion

annual_rows = []
for idx, row in temp_df.iterrows():
    decade_str = row["Decade"]
    start_year, end_year = [int(x) for x in decade_str.split("-")]
    for y in range(start_year, end_year + 1):
        annual_rows.append({
            "Year": y,
            "Mean_Temperature": row["Mean_Temperature"]
        })
annual_temp_df = pd.DataFrame(annual_rows)
print(annual_temp_df)

In [ ]:

import matplotlib.pyplot as plt

print(annual_temp_df.info())

plt.figure(figsize=(12, 6))
plt.plot(annual_temp_df["Year"], annual_temp_df["Mean_Temperature"], marker='o')
plt.title("Australia Mean Temperature (Yearly)")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Augmented Dickey-Fuller test for stationarity (trying statsmodels library for the first time)

from statsmodels.tsa.stattools import adfuller

result = adfuller(annual_temp_df["Mean_Temperature"])
print(f"ADF Statistic: {result[0]}")
print(f"p-value: {result[1]}")
if result[1] < 0.05:
    print("Series is stationary (reject null hypothesis)")
else:
    print("Series is not stacionary (fail to reject null hypothesis)")

In [ ]:
annual_temp_df["Diff_Temp"] = annual_temp_df["Mean_Temperature"].diff()

# Wanted to check ARIMA model
from statsmodels.tsa.arima.model import ARIMA

# ARIMA(1,1,0) already does first differencing internally
model_arima = ARIMA(annual_temp_df["Mean_Temperature"], order=(1,1,0)).fit()
future_pred_arima = model_arima.forecast(steps=50)
print(future_pred_arima)

In [ ]:
# New Index for forecast years
last_year = annual_temp_df.index.max() if annual_temp_df.index.name == "Year" else annual_temp_df["Year"].max()
forecast_years = range(last_year + 1, last_year + 1 + 50)
future_pred_arima.index = forecast_years

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(annual_temp_df["Year"], annual_temp_df["Mean_Temperature"], marker="o", label="Observed")
plt.plot(future_pred_arima.index, future_pred_arima, "r--", label="ARIMA Forecast")
plt.title("Australia Mean Temperature Forecast (ARIMA)")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
pred_obj = model_arima.get_forecast(steps=50)
ci = pred_obj.conf_int(alpha=0.05)
plt.figure(figsize=(14,6))
plt.plot(annual_temp_df["Year"], annual_temp_df["Mean_Temperature"], marker='o', label="Observed")
plt.plot(forecast_years, future_pred_arima, 'r--', label="ARIMA Forecast")
plt.fill_between(forecast_years, ci.iloc[:,0], ci.iloc[:,1], color='gray', alpha=0.3, label="95% CI")
plt.title("Australia Mean Temperature Forecast (ARIMA, with Confidence Intervals)")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()


> ⚠️ ARIMA did not produce a realistic forecast here due to the “stepwise” nature of the decadal input data and the strong upward trend.  
> For this type of data, **linear regression or exponential smoothing** will give more reliable future temperature projections.


#### Quick exploration for getting the overall trend.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

print(annual_temp_df)

X = annual_temp_df["Year"].values.reshape(-1, 1)
y = annual_temp_df["Mean_Temperature"].values

model = LinearRegression()
model.fit(X, y)

In [ ]:
# Forecast for the next 50 years
future_years = np.arange(annual_temp_df["Year"].max() + 1, annual_temp_df["Year"].max() + 51)
future_X = future_years.reshape(-1, 1)
future_pred = model.predict(future_X)
print(future_pred)

In [ ]:
# Plotting the linear regression forecast
plt.figure(figsize=(12, 6))
plt.plot(annual_temp_df["Year"], annual_temp_df["Mean_Temperature"], marker="o", label="Observed")
plt.plot(future_years, future_pred, "r--", label="Linear Regression Forecast")
plt.title("Australia Mean Temperature Forecast (Linear Regression)")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Show model parameters
print(f"Trend: {model.coef_[0]:.3f} °C/year")
print(f"Intercept: {model.intercept_:.2f}")

### 🔥 Predicting and Forecasting Australia’s Annual Mean Temperature

We use linear regression to model and forecast national mean temperature, first evaluating model accuracy with a train/test split, then using all available data for a future forecast.


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score


split_year = 2005
train = annual_temp_df[annual_temp_df["Year"] <= split_year]
test = annual_temp_df[annual_temp_df["Year"] > split_year]

X_train = train["Year"].values.reshape(-1, 1)
y_train = train["Mean_Temperature"].values
X_test = test["Year"].values.reshape(-1, 1)
y_test = test["Mean_Temperature"].values

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test R²: {r2:.3f}")


An R² of -6.044 means my model is not capturing the trend in the test data at all and is performing much worse than a naive mean prediction.

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(train["Year"], y_train, label="Train Observed")
plt.plot(test["Year"], y_test, label="Test Observed", marker="o")
plt.plot(test["Year"], y_pred, label="Test Predicted", marker="s", linestyle='--')
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.title("Actual vs Predicted Mean Temperature (Train/Test)")
plt.legend()
plt.tight_layout()
plt.show()

#### 📌 Why Predict Annual Mean Temperature Instead of Decadal Means?

In this notebook, we focus on predicting the **mean temperature for each year**, rather than using decadal averages. It’s turning out to be quite a bit more difficult than I expected to find the average annual temperature for each year. The average per decade is easier to find, but it provides much less precision for predictions.

In [ ]:
final_temp_df = pd.read_csv(os.path.join(TEMP_DATASET_DIR, "final_mean_temp.csv"))

print(final_temp_df)

In [ ]:
final_temp_df["Year"] = pd.to_datetime(final_temp_df["Year"], format="%Y")
final_temp_df = final_temp_df.set_index("Year")
final_temp_df = final_temp_df.sort_index() 
final_temp_df = final_temp_df.asfreq('YS')  # 'YS' = Year Start

In [ ]:
split_year = 2005  # Train on 1901-2005, test on 2006 onwards
train = final_temp_df.loc[:f"{split_year}"]
test = final_temp_df.loc[str(int(split_year) + 1):]

X = final_temp_df.index.year.values.reshape(-1, 1)
y = final_temp_df["Mean_Temperature"].values


In [ ]:
# Linear Regression model

X_train = train.index.year.values.reshape(-1, 1)
y_train = train["Mean_Temperature"].values
X_test = test.index.year.values.reshape(-1, 1)
y_test = test["Mean_Temperature"].values

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
# Metrics rmse and r2
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Linear Regression Test RMSE: {rmse:.3f}")
print(f"Linear Regression Test R2: {r2:.3f}")



In [ ]:
# Residuals
residuals = y_test - y_pred

# Plotting Residuals
plt.figure(figsize=(12, 6))
plt.scatter(test.index.year.values.reshape(-1, 1), residuals)
plt.axhline(0, color="gray", linestyle="--")
plt.title("Linear Regression Residuals (Test Set)")
plt.xlabel("Year")
plt.ylabel("Residual (°C)")
plt.tight_layout()
plt.show()

#### 📊 Linear Regression Results on Annual Mean Temperature

Using the new annual mean temperature dataset (`final_temp_df`), I trained a linear regression model to predict Australia’s national mean temperature. The model’s performance was evaluated using RMSE and R² metrics, as well as a residuals plot.

- **RMSE** quantifies the average prediction error in °C; lower values indicate better accuracy.
- **R²** shows how much of the temperature variation is explained by the model; values closer to 1 are better.
- The **residuals plot** helps check for patterns or outliers in the prediction errors.

These results provide insight into how well a simple linear trend captures Australia’s annual temperature changes and highlight areas where the model may be improved.

#### 📝 Conclusion: Linear Regression for Annual Mean Temperature

The linear regression model applied to the annual mean temperature data did not perform well, as indicated by a low (or negative) R² and a relatively high RMSE. This suggests that a simple linear trend is not sufficient to capture the complexity and variability in Australia’s yearly temperature changes. More advanced or flexible modeling approaches may be needed to improve prediction accuracy and better reflect the underlying climate dynamics.

In [ ]:
# Exponential Smoothing model (Holt's Linear Trend Method)
from statsmodels.tsa.holtwinters import ExponentialSmoothing

holt_model = ExponentialSmoothing(train["Mean_Temperature"], trend="add", seasonal=None).fit()
y_pred_holt = holt_model.forecast(len(test))

# Metrics for Holt's model
rmse_holt = np.sqrt(mean_squared_error(y_test, y_pred_holt))
r2_holt = r2_score(y_test, y_pred_holt)

print(f"Exp Smoothing Test RMSE: {rmse_holt:.3f}")
print(f"Exp Smoothing Test R2: {r2_holt:.3f}")


In [ ]:
# Residuals for Holt's model
residuals_holt = y_test - y_pred_holt

# Plotting Holt's model predictions
plt.figure(figsize=(12, 6))
plt.scatter(test.index.year.values.reshape(-1, 1), residuals_holt)
plt.axhline(0, color="gray", linestyle="--")
plt.title("Exponential Smoothing Residuals (Test Set)")
plt.xlabel("Year")
plt.ylabel("Residual (°C)")
plt.tight_layout()
plt.show()


#### 📊 Exponential Smoothing Results on Annual Mean Temperature

Using Holt’s Exponential Smoothing, the model achieved an RMSE of **0.379 °C** and an R² of **0.205** on the test set. This indicates a moderate prediction error and that about 20% of the year-to-year temperature variation is explained by the model. While this is an improvement over linear regression, it suggests that further enhancements or more advanced models may be needed to better capture the complexity of Australia’s annual temperature trends.

The residuals plot for Holt’s model shows how prediction errors vary by year. Most points are close to zero, but some years have larger errors, indicating the model captures the general trend but misses some year-to-year fluctuations.

In [ ]:
# ARIMA model
from statsmodels.tsa.arima.model import ARIMA 

arima_model = ARIMA(train["Mean_Temperature"], order=(1, 1, 0)).fit()
y_pred_arima = arima_model.forecast(steps=len(test))


In [ ]:
# Metrics for ARIMA model
rmse_arima = np.sqrt(mean_squared_error(y_test, y_pred_arima))
r2_arima = r2_score(y_test, y_pred_arima)

print(f"ARIMA Test RMSE: {rmse_arima:.3f}")
print(f"ARIMA Test R²: {r2_arima:.3f}")

In [ ]:
# Residuals and plotting
residuals_arima = y_test - y_pred_arima

plt.Figure(figsize=(12, 6))
plt.scatter(test.index.year.values.reshape(-1, 1), residuals_arima)
plt.axhline(0, color="gray", linestyle="--")
plt.title("ARIMA Residuals (Test Set)")
plt.xlabel("Year")
plt.ylabel("Residual (°C)")
plt.tight_layout()
plt.show()

#### 📊 ARIMA Model Results

The ARIMA model produced an RMSE that reflects the average prediction error in °C—lower values indicate more accurate forecasts. The R² value shows how much of the year-to-year temperature variation is explained by the model; values closer to 1 are better, while values near 0 or negative indicate a poor fit. The residuals plot displays the difference between actual and predicted temperatures for each year. Ideally, residuals should be randomly scattered around zero; patterns or large deviations suggest the model is missing some structure in the data.

In [ ]:
# Polynomial Regression model
from sklearn.preprocessing import PolynomialFeatures

degree = 2  # Degree 2 better after testing Degree 3 too

poly = PolynomialFeatures(degree=degree)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

model_poly = LinearRegression().fit(X_train_poly, y_train)
y_pred_poly = model_poly.predict(X_test_poly)

In [ ]:
# Metrics
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly = r2_score(y_test, y_pred_poly)

print(f"Poly deg {degree} Test RMSE: {rmse_poly:.3f}")
print(f"Poly deg {degree} Test R2: {r2_poly:.3f}")

In [ ]:
# Plotting Polynomial Regression predictions
plt.figure(figsize=(12,6))
plt.plot(train.index.year, y_train, label="Train Observed")
plt.plot(test.index.year, y_test, marker="o", label="Test Observed")
plt.plot(test.index.year, y_pred_poly, marker="s", linestyle='--', label="Poly Predicted")
plt.legend()
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.title(f"Polynomial Regression (Degree {degree})")
plt.tight_layout()
plt.show()

#### 📊 Polynomial Regression (Degree 2) Results

The degree 2 polynomial regression model achieved an RMSE of **0.381 °C** and an R² of **0.197** on the test set. This means the model’s average prediction error is similar to previous models, and it explains about 20% of the year-to-year temperature variation. The plot shows that while the polynomial model can capture some curvature in the trend, it still misses much of the variability in annual temperatures.

In [ ]:
# Random Forest Regressor model
from sklearn.ensemble import RandomForestRegressor

df = final_temp_df.copy()

X = df.index.year.values.reshape(-1, 1)
y = df["Mean_Temperature"].values 

split_year = 2005
X_train_rf = X[df.index.year <= split_year]
y_train_rf = y[df.index.year <= split_year]
X_test_rf = X[df.index.year > split_year]
y_test_rf = y[df.index.year > split_year]

In [ ]:
random_forest = RandomForestRegressor(n_estimators=100, random_state=42)
random_forest.fit(X_train_rf, y_train_rf)
y_pred_rf = random_forest.predict(X_test_rf)
# Metrics for Random Forest
rmse_rf = np.sqrt(mean_squared_error(y_test_rf, y_pred_rf))
r2_rf = r2_score(y_test_rf, y_pred_rf)
print(f"Random Forest Test RMSE: {rmse_rf:.3f}")
print(f"Random Forest Test R2: {r2_rf:.3f}")

In [ ]:
# Plotting Actual vs Predicted for Random Forest
plt.figure(figsize=(12,6))
plt.plot(X_train, y_train, label="Train Observed", color="gray")
plt.plot(X_test, y_test, 'o', label="Test Observed", color="black")
plt.plot(X_test, y_pred_rf, 's--', label="RF Predicted", color="darkgreen")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.title("Random Forest Regression: Australia Mean Temperature")
plt.legend()
plt.tight_layout()
plt.show()

#### 📊 Random Forest Regression Results

The Random Forest model gave an RMSE of **0.444 °C** and an R² of **-0.091** on the test set. This means its average prediction error is higher than other models, and the negative R² indicates it performs worse than simply predicting the mean. The plot shows that the model does not capture the year-to-year temperature trend well, with predictions often missing the actual values.

In [ ]:
# Let's back to ARIMA model, but checking the best parameters
from pmdarima import auto_arima

auto_arima_model = auto_arima(train["Mean_Temperature"], seasonal=False, stepwise=True, trace=True)
print(auto_arima_model.summary())

In [ ]:
arima_best = ARIMA(train["Mean_Temperature"], order=(0, 1, 1)).fit()
y_pred_auto_arima = arima_best.forecast(steps=len(test))

# Metrics
rmse_auto_arima = np.sqrt(mean_squared_error(y_test, y_pred_auto_arima))
r2_auto_arima = r2_score(y_test, y_pred_auto_arima)
print(f"Auto ARIMA Test RMSE: {rmse_auto_arima:.3f}")
print(f"Auto ARIMA Test R2: {r2_auto_arima:.3f}")

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(train.index.year, y_train, label="Train Observed")
plt.plot(test.index.year, y_test, marker="o", label="Test Observed")
plt.plot(test.index.year, y_pred_auto_arima, marker="s", linestyle='--', label="Auto ARIMA Predicted")
plt.legend()
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.title("Auto ARIMA Model: Australia Mean Temperature")
plt.tight_layout()
plt.show()

#### 📉 ARIMA(0,1,1) Model: Results and Interpretation

After automatic model selection with `auto_arima`, the best-fitting ARIMA model for Australia’s annual mean temperature was **ARIMA(0,1,1)**. This model is equivalent to a simple exponential smoothing approach, commonly used for climate time series with persistent trends.

- **Model performance on the test set (2006–2021):**
  - RMSE ≈ 0.47 °C
  - R² ≈ –0.21

**Interpretation:**  
The ARIMA(0,1,1) model, like other regression and smoothing models, was unable to accurately predict year-to-year temperature values on unseen data (negative R²). This result is expected for climate time series with strong long-term trends but large natural variability in individual years.

The plot below compares the observed test years with ARIMA(0,1,1) predictions.


#### 🔍 Comparing the Best Two Forecasting Models

After testing several forecasting methods—including linear regression, polynomial regression, exponential smoothing, ARIMA, and random forest—we selected the two best-performing models: polynomial regression (degree 2) and Holt’s exponential smoothing. 

The following comparison focuses on these two approaches, which provided the most accurate and robust forecasts for Australia’s annual mean temperature.


In [ ]:
# Plotting the best two models: Polynomial Regression and Holt's Exponential Smoothing
plt.figure(figsize=(14, 7))
plt.plot(df.index.year, df["Mean_Temperature"], color="black", label="Observed", linewidth=2)
plt.plot(test.index.year, y_pred_poly, "r--", label="Poly Regression (Degree 2)")
plt.plot(test.index.year, y_pred_holt, "g--", label="Holt's Exponential Smoothing")
plt.title("Comparison of Best Two Models: Annual Mean Temperature (Test Years)")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Polynomial Regression Forecast for the next 50 years
X_all = df.index.year.values.reshape(-1, 1)
poly_model_full = LinearRegression().fit(poly.fit_transform(X_all), df["Mean_Temperature"])
future_period = np.arange(df.index.year.max() + 1, df.index.year.max() + 51)
X_future_poly = poly.transform(future_period.reshape(-1, 1))
future_pred_poly = poly_model_full.predict(X_future_poly)

# Holt's Exponential Smoothing Forecast for the next 50 years
holt_model_full = ExponentialSmoothing(df["Mean_Temperature"], trend="add", seasonal=None).fit()
future_pred_holt = holt_model_full.forecast(50)

In [ ]:
# Plotting
plt.figure(figsize=(15, 7))
plt.plot(df.index.year, df["Mean_Temperature"], color="black", label="Observed", linewidth=2)
plt.plot(future_period, future_pred_poly, "r--", label="Poly Regression Forecast")
plt.plot(future_period, future_pred_holt, "g--", label="Holt's Exponential Smoothing Forecast")
plt.title("Forecasting Australia's Mean Temperature for the Next 50 Years")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Poly Regression Predictions into a DataFrame
future_poly_df = pd.DataFrame({
    "Year": future_period,
    "Mean_Temperature": future_pred_poly
})

print(future_poly_df)

In [ ]:
# Holt's Exponential Smoothing Predictions into a DataFrame
future_holt_df = pd.DataFrame({
    "Year": future_period,
    "Mean_Temperature": future_pred_holt
})

print(future_holt_df)

#### Results and Best Models

#### 📊 Model Comparison and Forecast Interpretation

Both polynomial regression and Holt’s exponential smoothing models project continued warming in Australia’s national mean temperature over the next 50 years. The polynomial model (degree 2) forecasts a slightly faster, accelerating rise compared to the smoother trend from exponential smoothing. 

**Key takeaways:**
- Both models show a clear, ongoing increase, with projected temperatures by 2070–2075 rising well above the current mean.
- This result is consistent with Australian Bureau of Meteorology (BoM) and CSIRO climate projections.
- The differences between the models illustrate the plausible range of future warming, depending on whether trends accelerate (polynomial) or continue steadily (smoothing).

> This evidence supports a strong risk of crossing key habitability thresholds (e.g., extreme heat, water stress) by mid-century, underscoring the urgency of climate adaptation and mitigation.



In [ ]:
# Merge both predictions into a single DataFrame and CSV
temperature_predictions_df = pd.merge(future_holt_df, future_poly_df, on="Year", suffixes=("_Holt", "_Poly"))
print(temperature_predictions_df)

import os
MLRESULTS_DIR = r"d:\PersonalProjects\WeatherPredictionAustralia\ml_results"
temperature_predictions_df.to_csv(os.path.join(MLRESULTS_DIR, "temp", "csv_files", "temperature_predictions.csv"), index=False)

#### 🌡️ Habitability Thresholds in Annual Mean Temperature

We include scientific “risk thresholds” for annual mean temperature:
- **27°C**: Onset of severe food, water, and health risks in most regions.
- **29°C**: Upper edge of the “human climate niche”; chronic heat stress, extreme risk for sustained habitability (Xu et al., 2020, PNAS).
- **30°C**: Nearly uninhabited in the modern era.

These thresholds are visualized in our forecast to highlight when Australia could face unprecedented habitability risks.


In [ ]:
# Let's add habitability thresholds in annual mean temperature
plt.figure(figsize=(15, 7))
plt.plot(temperature_predictions_df["Year"], temperature_predictions_df["Mean_Temperature_Holt"], color="black", label="Observed", linewidth=2)
plt.plot(temperature_predictions_df["Year"], temperature_predictions_df["Mean_Temperature_Poly"], color="black", label="Observed", linewidth=2)
plt.plot(future_period, future_pred_poly, "r--", label="Poly Regression Forecast")
plt.plot(future_period, future_pred_holt, "g--", label="Holt's Exponential Smoothing Forecast")

plt.axhline(27, color="orange", linestyle="--", label="27°C Threshold")
plt.axhline(29, color='red', linestyle='--', label='29°C: Habitability limit')
plt.axhline(30, color='purple', linestyle='-.', label='30°C: Extreme danger')

plt.title("Forecast with Habitability Thresholds")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(MLRESULTS_DIR, "temp", "plots", "temp_forecast_next50years_with_thresholds.png"), dpi=300)
plt.show()

In [ ]:
# Let's add habitability thresholds in csv
def habitability_thresholds(row):
    if row["Mean_Temperature_Poly"] >= 30 or row["Mean_Temperature_Holt"] >= 30:
        return "Extreme Danger"
    elif row["Mean_Temperature_Poly"] >= 29 or row["Mean_Temperature_Holt"] >= 29:
        return "Habitability Risk"
    elif row["Mean_Temperature_Poly"] >= 27 or row["Mean_Temperature_Holt"] >= 27:
        return "Food/Water Risk"
    else:
        return ""
    
temperature_predictions_df["Habitability_Threshold"] = temperature_predictions_df.apply(habitability_thresholds, axis=1)
temperature_predictions_df.to_csv(os.path.join(MLRESULTS_DIR, "temp", "csv_files", "temperature_predictions_with_thresholds.csv"), index=False)
print(temperature_predictions_df)

In [ ]:
# Extend future period to 100 years
future_period_extended = np.arange(temperature_predictions_df["Year"].max() + 1, 2122)

# Input for models
X_future_poly_extended = poly.transform(future_period_extended.reshape(-1, 1))
future_pred_poly_extended = poly_model_full.predict(X_future_poly_extended)

future_pred_holt_extended = holt_model_full.forecast(len(future_period_extended))

# Merge extended predictions into a DataFrame
future_predictions_extended_df = pd.DataFrame({
    "Year": future_period_extended,
    "Mean_Temperature_Holt": future_pred_holt_extended,
    "Mean_Temperature_Poly": future_pred_poly_extended
})

def habitability_thresholds(row):
    if row["Mean_Temperature_Holt"] >= 30 or row["Mean_Temperature_Poly"] >= 30:
        return "Extreme Danger"
    elif row["Mean_Temperature_Holt"] >= 29 or row["Mean_Temperature_Poly"] >= 29:
        return "Habitability Risk"
    elif row["Mean_Temperature_Holt"] >= 27 or row["Mean_Temperature_Poly"] >= 27:
        return "Food/Water Risk"
    else:
        return ""
    
future_predictions_extended_df["Habitability_Threshold"] = future_predictions_extended_df.apply(habitability_thresholds, axis=1)
future_predictions_extended_df.to_csv(os.path.join(MLRESULTS_DIR, "temp", "csv_files", "temperature_predictions_100years_with_thresholds.csv"), index=False)



print(future_predictions_extended_df)

In [ ]:
# Plotting next 100 years with habitability thresholds
plt.figure(figsize=(15, 7))
plt.plot(future_predictions_extended_df["Year"], future_predictions_extended_df["Mean_Temperature_Holt"], color="black", label="Observed", linewidth=2)
plt.plot(future_predictions_extended_df["Year"], future_predictions_extended_df["Mean_Temperature_Poly"], color="black", label="Observed", linewidth=2)
plt.plot(future_period_extended, future_pred_poly_extended, "r--", label="Poly Regression Forecast")
plt.plot(future_period_extended, future_pred_holt_extended, "g--", label="Holt's Exponential Smoothing Forecast")

# Thresholds
plt.axhline(27, color="orange", linestyle=":", label="27°C: Risk Threshold")
plt.axhline(29, color="red", linestyle=":", label="29°C: Habitability Limit")
plt.axhline(30, color="purple", linestyle=":", label="30°C: Extreme Danger")

plt.title("Forecast to 2121: Australia Mean Temperature")
plt.xlabel("Year")
plt.ylabel("Mean Temperature (°C)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(MLRESULTS_DIR, "temp", "plots", "temperature_next_100_years_with_thresholds.png"), dpi=300)
plt.show()


### 🚦 Final Temperature Projection:

Both polynomial regression and exponential smoothing forecast a strong rise in Australia’s national mean temperature over the next 100 years, reaching ~26°C by 2121. 

**Key risk thresholds (27°C, 29°C, 30°C)**—used in global climate research to define extreme danger for human life—are not crossed within this forecast period under a simple continuation of historical trends.

> **Interpretation:**  
> While Australia faces serious and escalating risks from heat, drought, bushfire, and infrastructure breakdown, national mean temperature alone is unlikely to render the continent completely uninhabitable by 2121.  
> However, local and regional uninhabitability (especially in the interior and far north) may occur much sooner, and habitability must be assessed in combination with rainfall, water stress, and climate extremes.
